In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh37')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe014.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
mt = genotypes.get_ukb_imputed_v3_bgen([21])

2021-10-27 11:21:38 Hail: INFO: Number of BGEN files parsed: 1
2021-10-27 11:21:38 Hail: INFO: Number of samples in BGEN files: 487409
2021-10-27 11:21:38 Hail: INFO: Number of variants across all BGEN files: 1261158


In [24]:
ht = genotypes.get_ukb_imputed_v3_mfi(21)

2021-10-27 11:30:58 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)


----------------------------------------
Global fields:
    None
----------------------------------------
Row fields:
    'varid': str 
    'rsid': str 
    'position': int32 
    'a1': str 
    'a2': str 
    'maf': float64 
    'f6': str 
    'info': float64 
----------------------------------------
Key: ['rsid']
----------------------------------------


2021-10-27 11:31:02 Hail: INFO: Ordering unsorted dataset with network shuffle


,
rsid,varid
str,str
"""21:10000025_TA_T""","""21:10000025_TA_T"""
"""21:10000037_TG_T""","""21:10000037_TG_T"""
"""21:10000064_AT_A""","""21:10000064_AT_A"""
"""21:10021066_AT_A""","""21:10021066_AT_A"""
"""21:10024632_G_T""","""21:10024632_G_T"""
"""21:10026688_TC_T""","""21:10026688_TC_T"""
"""21:10027345_G_T""","""21:10027345_G_T"""
"""21:10034125_AG_A""","""21:10034125_AG_A"""


In [25]:
ht = ht.annotate(variant = hl.delimit([hl.str(21), hl.str(ht.position), ht.a1, ht.a2], ':'))
ht = ht.key_by(**hl.parse_variant(ht.variant))
ht = variants.liftover_hail_table(ht)
ht.select('info', 'rsid').describe()

AttributeError: Table instance has no field, method, or property 'locus'
    Hint: use 'describe()' to show the names of all data fields.

2021-10-27 11:23:06 Hail: ERROR: Analysis exception: 'Table.select': cannot overwrite key field 'alleles' with annotate, select or drop; use key_by to modify keys.


ExpressionException: 'Table.select': cannot overwrite key field 'alleles' with annotate, select or drop; use key_by to modify keys.

In [11]:
final_sample_list = "/well/lindgren/UKBIOBANK/dpalmer/wes_200k/ukb_wes_qc/data/samples/09_final_qc.keep.sample_list"
ht_final_samples = hl.import_table(final_sample_list, no_header=True, key='f0', delimiter = ',')

2021-10-26 15:43:51 Hail: INFO: Loading 127 fields. Counts by type:
  str: 127


In [ ]:
ht_final_samples

In [16]:
lst = (hl.is_defined(ht_final_samples[mt.col_key])).collect()

In [17]:
sum(lst)

176614

In [7]:
mt = mt.filter_cols(hl.is_defined(ht_final_samples[mt.col_key]))
mt.count()

(3343729, 0)

In [31]:
def get_ukb_imputed_v3_sample_path(chrom, original: bool = False):
    """Get path to UK Biobank imputed v3 genotypes sample files
    :param chrom: Which chromosome to get (autosome, X and XY)
    """
    if chrom in range(1,23):
        chrom = 1
    return f"/well/lindgren/UKBIOBANK/DATA/SAMPLE_FAM/ukb11867_imp_chr{chrom}_v3_s487395.sample"

In [30]:
def overlap(a, b):
    return list(set(a) & set(b))

In [29]:
genotypes.AUTOSOMES
CHROMOSOMES = genotypes.AUTOSOMES + ['X', 'XY']

[1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 'X',
 'XY']

In [19]:
def get_ukb_imputed_v3_bgen(chroms: list, entry_fields: list = ['GT', 'GP', 'dosage']):
    """Import UK Biobank imputed v3 genotypes as a MatrixTable from bgens
    :param chroms: List of chromosomes to import
    :param entry_fields: List of entry fields to include. Default: ['GT', 'GP', 'dosage']
    """
    
    bgen_autosomes_path_list = list(map(genotypes.get_ukb_imputed_v3_bgen_path, overlap(chroms, AUTOSOMES)))
    
    bgen_chrom_xy_path = genotypes.get_ukb_imputed_v3_bgen_path, 
    bgen_path_list = bgen_autosomes_path_list + bgen_sex_chrom_path_list
    
    for chrom, bgen_path in zip(chroms, bgen_path_list):
        if not hl.hadoop_is_dir(bgen_path+".idx2"):
            hl.index_bgen(
                path=bgen_path,
                reference_genome="GRCh37",
                contig_recoding={str(chrom).zfill(2): str(chrom)},
            )
    
    
    mt = hl.import_bgen(
        path=bgen_path_list,
        entry_fields=entry_fields,
        sample_file="/well/lindgren/UKBIOBANK/DATA/SAMPLE_FAM/ukb11867_imp_chr1_v3_s487395.sample"
    )

    return mt

SyntaxError: invalid syntax (<ipython-input-19-45be7a681dbf>, line 16)

['/well/lindgren/UKBIOBANK/nbaya/resources/ref/ukb_wes_200k/ukb_ref_panel/data/imputed_v3_bgen/ukb_imp_chrX_v3.bgen']

In [7]:
hl.import_bgen(
    path="/well/lindgren/UKBIOBANK/DATA/IMPUTATION/ukb_imp_chrX_v3.bgen",
    entry_fields=['GT'],
    sample_file="/well/lindgren/UKBIOBANK/DATA/SAMPLE_FAM/ukb11867_imp_chr1_v3_s487395.sample"
)

FatalError: HailException: The following BGEN files have no .idx2 index file. Use 'index_bgen' to create the index file once before calling 'import_bgen':
  file:/well/lindgren/UKBIOBANK/DATA/IMPUTATION/ukb_imp_chrX_v3.bgen)

Java stack trace:
is.hail.utils.HailException: The following BGEN files have no .idx2 index file. Use 'index_bgen' to create the index file once before calling 'import_bgen':
  file:/well/lindgren/UKBIOBANK/DATA/IMPUTATION/ukb_imp_chrX_v3.bgen)
	at is.hail.utils.ErrorHandling.fatal(ErrorHandling.scala:11)
	at is.hail.utils.ErrorHandling.fatal$(ErrorHandling.scala:11)
	at is.hail.utils.package$.fatal(package.scala:78)
	at is.hail.io.bgen.LoadBgen$.getIndexFiles(LoadBgen.scala:247)
	at is.hail.io.bgen.MatrixBGENReader$.apply(LoadBgen.scala:321)
	at is.hail.io.bgen.MatrixBGENReader$.fromJValue(LoadBgen.scala:303)
	at is.hail.expr.ir.MatrixReader$.fromJson(MatrixIR.scala:87)
	at is.hail.expr.ir.IRParser$.matrix_ir_1(Parser.scala:1704)
	at is.hail.expr.ir.IRParser$.$an